<a href="https://colab.research.google.com/github/Janith303/Sinhala-and-Maths-Model-training-/blob/Resnet50/notebooks/ResNet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys
import json
import hashlib
import time
import random
import platform
import shutil
import subprocess
import gc

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import sklearn
import kagglehub
import matplotlib
import matplotlib.pyplot as plt

# Architecture-specific preprocessor for ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet50_preprocess_input

print("All imports completed successfully.")

All imports completed successfully.


In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    determinism_status = "TensorFlow op determinism enabled"
except Exception as e:
    determinism_status = f"Could not enable op determinism: {e}"

print(f"Seed:        {SEED}")
print(f"Determinism: {determinism_status}")

Seed:        42
Determinism: TensorFlow op determinism enabled


In [3]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        gpu_name = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            text=True,
        ).strip().splitlines()[0]
        nvidia_smi = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'],
            text=True,
        ).strip().splitlines()[0]
    except Exception:
        gpu_name = tf.config.experimental.get_device_details(gpus[0]).get('device_name', 'Unknown GPU')
        nvidia_smi = 'Unavailable'
else:
    gpu_name = 'No GPU detected'
    nvidia_smi = 'N/A'

environment_info = {
    'member': 3,
    'model': 'ResNet50',
    'python_version': sys.version,
    'platform': platform.platform(),
    'tensorflow_version': tf.__version__,
    'keras_version': keras.__version__,
    'numpy_version': np.__version__,
    'pandas_version': pd.__version__,
    'scikit_learn_version': sklearn.__version__,
    'kagglehub_version': getattr(kagglehub, '__version__', 'unknown'),
    'matplotlib_version': matplotlib.__version__,
    'gpu_name': gpu_name,
    'nvidia_smi': nvidia_smi,
    'seed': SEED,
    'determinism_status': determinism_status,
}

print('Environment Recorded:')
for k, v in environment_info.items():
    print(f'  {k:<22}: {v}')

Environment Recorded:
  member                : 3
  model                 : ResNet50
  python_version        : 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
  platform              : Linux-6.6.122+-x86_64-with-glibc2.39
  tensorflow_version    : 2.21.0
  keras_version         : 3.13.2
  numpy_version         : 2.1.3
  pandas_version        : 2.2.3
  scikit_learn_version  : 1.6.1
  kagglehub_version     : 1.0.2
  matplotlib_version    : 3.10.0
  gpu_name              : No GPU detected
  nvidia_smi            : N/A
  seed                  : 42
  determinism_status    : TensorFlow op determinism enabled


In [4]:
from google.colab import drive
drive.mount("/content/drive")


MessageError: Error: credential propagation was unsuccessful

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Dataset")
DATA_DIR = PROJECT_DIR / "data" / "raw"
SPLITS_DIR = PROJECT_DIR / "data" / "splits"
RESULTS_DIR = PROJECT_DIR / "results"
PLOTS_DIR = RESULTS_DIR / "plots"
METRICS_DIR = RESULTS_DIR / "metrics"
MODELS_DIR = PROJECT_DIR / "models"
FINAL_MODEL_DIR = MODELS_DIR / "final"
CHECKPOINT_DIR = MODELS_DIR / "checkpoints"

for folder in [
    SPLITS_DIR, PLOTS_DIR, METRICS_DIR, FINAL_MODEL_DIR, CHECKPOINT_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet50_preprocess_input

# Standard input size for ResNet50
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32

def process_path(file_path, label):
    """
    Reads an image file, converts it to pure black and white,
    and applies ResNet50 preprocessing.
    """
    # 1. Load the raw image
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3) # Adjust to decode_png if your scans are PNGs

    # 2. Resize to ResNet50 standard input shape
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])

    # 3. Convert to grayscale
    gray = tf.image.rgb_to_grayscale(img)

    # 4. Apply Binary Thresholding (Convert to pure Black and White)
    # Pixel values > 127 become 255 (white), others become 0 (black)
    bw_image = tf.where(gray > 127.0, 255.0, 0.0)

    # 5. ResNet50 requires 3 color channels. We tile the 1-channel B&W image into 3 identical channels.
    bw_image_3c = tf.tile(bw_image, [1, 1, 3])

    # 6. Apply ResNet50 ImageNet preprocessing (Zero-centers the tensor)
    # Ensure it's cast to float32 before preprocessing
    processed_img = resnet50_preprocess_input(tf.cast(bw_image_3c, tf.float32))

    return processed_img, label

def create_dataset(image_paths, labels, is_training=True):
    """
    Constructs an optimized tf.data.Dataset pipeline.
    """
    # Create a dataset of file paths and labels
    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))

    if is_training:
        # Shuffle the training data
        ds = ds.shuffle(buffer_size=len(image_paths), seed=42)

    # Apply the preprocessing function across multiple CPU cores
    ds = ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)

    # Batch and prefetch for GPU optimization
    ds = ds.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

    return ds

# --- Example Usage for your Colab Environment ---
# Assuming you have a list of file paths and integer labels:
# train_paths = [str(DATA_DIR / "class_0/img1.jpg"), str(DATA_DIR / "class_1/img2.jpg")]
# train_labels = [0, 1]
# train_dataset = create_dataset(train_paths, train_labels, is_training=True)